In [0]:
import requests

endpoint = "your-url.trycloudflare.com"

response = requests.get(
    f"{endpoint}/minio/health/live"
)

print(response.status_code)
print(response.text)

200



In [0]:
import boto3
from botocore.client import Config
import pandas as pd
import io

# Configure boto3 to connect to MinIO
s3_client = boto3.client(
    's3',
    endpoint_url=endpoint,
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    verify=True
)

# Test connection by listing buckets
try:
    buckets = s3_client.list_buckets()
    print(f"Successfully connected to MinIO at {endpoint}")
    print(f"Buckets: {[b['Name'] for b in buckets['Buckets']]}")
except Exception as e:
    print(f"Failed to connect: {e}")

# List objects in the target path
prefix = "raw/idfm/dataset=arrets/"
response = s3_client.list_objects_v2(Bucket='idfm-data', Prefix=prefix, MaxKeys=10)

if 'Contents' in response:
    print(f"\nFound {len(response['Contents'])} objects (showing first 10):")
    parquet_files = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.parquet')]
    
    if parquet_files:
        # Read the first parquet file as an example
        first_file = parquet_files[0]
        print(f"\nReading: {first_file}")
        
        obj = s3_client.get_object(Bucket='idfm-data', Key=first_file)
        parquet_data = obj['Body'].read()
        
        # Read parquet data into pandas DataFrame
        df_pandas = pd.read_parquet(io.BytesIO(parquet_data))
        
        # Convert to Spark DataFrame
        df_raw = spark.createDataFrame(df_pandas)

        print(df_raw.printSchema())
        
        print(f"\nDataFrame shape: {df_pandas.shape}")
        print(df_raw.display())
    else:
        print("No parquet files found in the specified path")
else:
    print(f"No objects found with prefix: {prefix}")

Successfully connected to MinIO at https://assess-player-the-prisoners.trycloudflare.com
Buckets: ['idfm-data']

Found 1 objects (showing first 10):

Reading: raw/idfm/dataset=arrets/ingestion_date=2026-08-25/data.parquet
root
 |-- arrid: string (nullable = true)
 |-- arrversion: string (nullable = true)
 |-- arrcreated: string (nullable = true)
 |-- arrchanged: string (nullable = true)
 |-- arrname: string (nullable = true)
 |-- arrtype: string (nullable = true)
 |-- arrxepsg2154: long (nullable = true)
 |-- arryepsg2154: long (nullable = true)
 |-- arrtown: string (nullable = true)
 |-- arrpostalregion: string (nullable = true)
 |-- arraccessibility: string (nullable = true)
 |-- arraudiblesignals: string (nullable = true)
 |-- arrvisualsigns: string (nullable = true)
 |-- arrfarezone: string (nullable = true)
 |-- zdaid: string (nullable = true)
 |-- arrgeopoint: struct (nullable = true)
 |    |-- lat: double (nullable = true)
 |    |-- lon: double (nullable = true)

None

DataFrame

arrid,arrversion,arrcreated,arrchanged,arrname,arrtype,arrxepsg2154,arryepsg2154,arrtown,arrpostalregion,arraccessibility,arraudiblesignals,arrvisualsigns,arrfarezone,zdaid,arrgeopoint
480701,1300789-1300791,2021-02-24T00:00:00+00:00,2026-08-24T14:51:57+00:00,Hôpital Jacques Cartier,bus,647863,6848247,Massy,91377,true,unknown,unknown,4,44219,"List(48.73224115795837, 2.291114726197896)"
497197,1835085-1835080,2026-08-24T14:02:35+00:00,2026-08-24T14:03:56+00:00,Kerautret - Salengro,bus,659532,6864601,Romainville,93063,unknown,unknown,unknown,3,497196,"List(48.88015580928261, 2.448213603604033)"
497202,1835101-1817493,2026-08-24T14:02:43+00:00,2026-08-24T14:03:56+00:00,Bergeries,bus,659866,6865874,Noisy-le-Sec,93053,unknown,unknown,unknown,3,496777,"List(48.89162448844716, 2.4526472731228632)"
497203,1835105-1817473,2026-08-24T14:02:45+00:00,2026-08-24T14:03:56+00:00,Cimetière de Noisy,bus,659951,6866165,Noisy-le-Sec,93053,unknown,unknown,unknown,3,496771,"List(48.89424665648571, 2.4537790242705966)"
490548,1835125-1522369,2023-12-22T17:46:40+00:00,2026-08-24T13:28:41+00:00,Centre,bus,725457,6851328,Lescherolles,77247,false,unknown,unknown,5,490549,"List(48.76155471658852, 3.3463191372163803)"
490550,1835122-1522369,2023-12-22T17:46:41+00:00,2026-08-24T13:28:05+00:00,Centre,bus,725417,6851361,Lescherolles,77247,false,unknown,unknown,5,490549,"List(48.76185309119053, 3.345776946738442)"
33690,1806082-1806083,2014-12-29T00:00:00+00:00,2026-08-24T12:29:50+00:00,Écluse,bus,683733,6803761,Moret-Loing-et-Orvanne,77316,false,unknown,unknown,5,55353,"List(48.33396203882058, 2.7805003684856193)"
497193,1835055-1575923,2026-08-24T10:14:57+00:00,2026-08-24T10:15:15+00:00,Saint-Denis - Pleyel,bus,651936,6868726,Saint-Denis,93066,unknown,unknown,unknown,2,46077,"List(48.9167279700141, 2.3441784174670035)"
14540,1835019-1835020,2014-12-29T00:00:00+00:00,2026-08-24T07:58:25+00:00,Gare de Villiers-le-Bel - Gonesse - Arnouville,bus,657209,6877278,Arnouville,95019,false,false,false,4,473620,"List(48.99400103715606, 2.41526032412292)"
12658,12658-53365,2014-12-29T15:31:51+00:00,2026-08-21T15:28:19+00:00,Collège Tillon,bus,647866,6825131,Lardy,91330,false,unknown,unknown,5,53365,"List(48.52432181193002, 2.2939657062622243)"


None


In [0]:
from datetime import datetime
import pandas as pd

# Convert to pandas FIRST to avoid any Spark metadata
df_bronze_pandas = df_pandas.copy()

# Add all metadata columns in pandas
df_bronze_pandas['_source'] = 'IDFM'
df_bronze_pandas['_ingestion_timestamp'] = pd.Timestamp.now()
df_bronze_pandas['_ingestion_date'] = pd.Timestamp.now().date()


# Write to parquet in memory
buffer = io.BytesIO()
df_bronze_pandas.to_parquet(buffer, index=False)
buffer.seek(0)

# Upload to MinIO using boto3
ingestion_date = datetime.now().strftime('%Y-%m-%d')
object_key = f"bronze/idfm/arrets/_ingestion_date={ingestion_date}/data.parquet"

try:
    s3_client.put_object(
        Bucket='idfm-data',
        Key=object_key,
        Body=buffer.getvalue()
    )
    print(f"Successfully uploaded to MinIO: {object_key}")
    print(f"Rows written: {len(df_bronze_pandas)}")
except Exception as e:
    print(f"Failed to upload: {e}")

    

Successfully uploaded to MinIO: bronze/idfm/arrets/_ingestion_date=2026-08-26/data.parquet
Rows written: 10000
